# 🧬 DNA Mixture NoC Classification - Kaggle Training

This notebook trains a neural network to predict **Number of Contributors (NoC)** in DNA mixtures using PROVEDIt data.

**What it does:**
- Loads pre-extracted DNA mixture features
- Trains a fully-connected neural network
- Evaluates on test set
- Generates visualizations and metrics

**Expected runtime:** ~2-3 minutes on GPU

In [ ]:
# Cell 1: Install Dependencies
!pip install -q torch scikit-learn pandas numpy scipy matplotlib seaborn > /dev/null 2>&1

import sys
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

print("✅ Dependencies installed")

## Step 1: Verify Dataset Path Configuration

In [ ]:
# Cell 2: Find Correct Dataset Path

def find_dataset_path():
    """Dynamically locate the dataset in Kaggle environment"""
    
    # Possible dataset paths
    possible_paths = [
        '/kaggle/input/datasets/honggiangtrnh/dnanet-noc-prediction',
        '/kaggle/input/dnanet-noc-prediction',
        '/kaggle/input/dnanet-noc-mixture',
    ]
    
    print("🔍 Searching for dataset...")
    
    # Check each possible path
    for path in possible_paths:
        if os.path.exists(path):
            print(f"✅ Found dataset at: {path}")
            return path
    
    # If not found, list available inputs
    print("❌ Dataset not found at standard paths")
    print("\n📂 Available in /kaggle/input/:")
    if os.path.exists('/kaggle/input'):
        for item in os.listdir('/kaggle/input'):
            item_path = os.path.join('/kaggle/input', item)
            if os.path.isdir(item_path):
                print(f"  - {item}/")
                # List subdirectories
                try:
                    for subitem in os.listdir(item_path):
                        if os.path.isdir(os.path.join(item_path, subitem)):
                            print(f"    - {subitem}/")
                except:
                    pass
    
    raise FileNotFoundError("Could not locate dataset. Please verify it's added to this notebook.")

# Find and set dataset path
DATASET_PATH = find_dataset_path()
print(f"\n📊 Dataset path: {DATASET_PATH}")

## Step 2: Check File Existence and Paths

In [ ]:
# Cell 3: Verify Required Files

required_files = {
    'features': 'features.npy',
    'labels': 'labels.npy',
    'noc_labels': 'noc_labels.csv',
    'feature_names': 'feature_names.json'
}

print("📋 Checking required files:")
print("=" * 50)

files_found = {}
for name, filename in required_files.items():
    filepath = os.path.join(DATASET_PATH, filename)
    exists = os.path.exists(filepath)
    status = "✅" if exists else "❌"
    
    if exists:
        file_size = os.path.getsize(filepath)
        size_mb = file_size / (1024**2)
        print(f"{status} {name:15} {filename:25} ({size_mb:.2f} MB)")
        files_found[name] = filepath
    else:
        print(f"{status} {name:15} {filename:25} NOT FOUND")

print("=" * 50)

# List all files in dataset
print(f"\n📂 Files in {DATASET_PATH}:")
dataset_files = os.listdir(DATASET_PATH)
for f in sorted(dataset_files)[:15]:  # Show first 15
    fpath = os.path.join(DATASET_PATH, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1024
        print(f"  - {f} ({size:.1f} KB)")

if len(dataset_files) > 15:
    print(f"  ... and {len(dataset_files) - 15} more files")

# Verify all required files found
if len(files_found) < len(required_files):
    missing = set(required_files.keys()) - set(files_found.keys())
    raise FileNotFoundError(f"Missing files: {missing}")

print(f"\n✅ All required files found!")
print(f"   Dataset location: {DATASET_PATH}")

## Step 3: Load and Validate Data

In [ ]:
# Cell 4: Load Data with Error Handling

def load_data_safely(dataset_path):
    """Load data with comprehensive error handling"""
    
    print("📦 Loading data...")
    
    try:
        # Load features
        features_path = os.path.join(dataset_path, 'features.npy')
        features = np.load(features_path)
        print(f"✅ Features loaded: shape {features.shape}")
        
        # Load labels
        labels_path = os.path.join(dataset_path, 'labels.npy')
        labels = np.load(labels_path)
        print(f"✅ Labels loaded: shape {labels.shape}")
        
        # Load feature names
        feature_names_path = os.path.join(dataset_path, 'feature_names.json')
        with open(feature_names_path, 'r') as f:
            feature_names = json.load(f)
        print(f"✅ Feature names loaded: {len(feature_names)} features")
        
        # Validate data
        print(f"\n📊 Data Validation:")
        print(f"  - Features shape: {features.shape}")
        print(f"  - Labels shape: {labels.shape}")
        print(f"  - Feature range: [{features.min():.3f}, {features.max():.3f}]")
        print(f"  - Label range: {labels.min()}-{labels.max()}")
        
        # Check shape compatibility
        assert features.shape[0] == labels.shape[0], \
            f"Shape mismatch: {features.shape[0]} samples vs {labels.shape[0]} labels"
        assert features.shape[1] == len(feature_names), \
            f"Feature count mismatch: {features.shape[1]} features vs {len(feature_names)} names"
        
        print(f"\n✅ All validation checks passed!")
        
        return features, labels, feature_names
        
    except FileNotFoundError as e:
        print(f"❌ File not found: {e}")
        print(f"📂 Make sure the dataset is added to this notebook")
        raise
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        raise

# Load data
features, labels, feature_names = load_data_safely(DATASET_PATH)

# Show NoC distribution
print(f"\n🧬 NoC Distribution:")
noc_counts = np.bincount(labels, minlength=11)[1:]
for noc in range(1, 11):
    count = noc_counts[noc-1]
    bar = "█" * count
    print(f"  NoC {noc:2d}: {count:2d} samples {bar}")

## Step 4: Train Neural Network

In [ ]:
# Cell 5: Setup Training

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
BATCH_SIZE = 8
EPOCHS = 50
LEARNING_RATE = 0.001
TEST_SIZE = 0.2
VAL_SPLIT = 0.2

print(f"⚙️  Configuration:")
print(f"  Device: {DEVICE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")

# Set seeds
np.random.seed(SEED)
torch.manual_seed(SEED)

# Convert labels (1-10 → 0-9 for PyTorch)
labels_pytorch = labels - 1

# Normalize features
print(f"\n🔄 Preprocessing data...")
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    features_scaled, labels_pytorch,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labels_pytorch
)

# Further split train into train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=y_train
)

print(f"  Train: {X_train.shape[0]} samples")
print(f"  Val:   {X_val.shape[0]} samples")
print(f"  Test:  {X_test.shape[0]} samples")

# Create data loaders
train_dataset = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).long()
)
val_dataset = TensorDataset(
    torch.from_numpy(X_val).float(),
    torch.from_numpy(y_val).long()
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test).long()
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"✅ Data preprocessing complete")

In [ ]:
# Cell 6: Define and Train Model

class NoCClassifier(nn.Module):
    """Fully-connected neural network for NoC classification"""
    
    def __init__(self, input_dim=78, hidden_dim=128, num_classes=10, dropout_rate=0.2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(hidden_dim, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        return x

# Build model
print(f"🏗️  Building model...")
model = NoCClassifier(input_dim=features.shape[1])
model = model.to(DEVICE)

print(f"  Model architecture:")
print(f"  Input: 78 features")
print(f"  Hidden: 128 → 64 neurons")
print(f"  Output: 10 classes (NoC 1-10)")
print(f"  Dropout: 0.2")

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training loop with early stopping
print(f"\n🚀 Training on {DEVICE}...")
print(f"{'Epoch':<8} {'Train Loss':<12} {'Val Loss':<12} {'Val Acc':<10}")
print("-" * 45)

history = {
    'train_loss': [],
    'val_loss': [],
    'val_acc': [],
    'best_epoch': 0,
    'best_val_acc': 0.0
}

patience_counter = 0
patience = 10

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_x.shape[0]
    
    train_loss /= len(train_loader.dataset)
    history['train_loss'].append(train_loss)
    
    # Validate
    model.eval()
    val_loss = 0.0
    val_correct = 0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            val_loss += loss.item() * batch_x.shape[0]
            
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == batch_y).sum().item()
    
    val_loss /= len(val_loader.dataset)
    val_acc = val_correct / len(val_loader.dataset)
    
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print progress
    if (epoch + 1) % max(1, EPOCHS // 10) == 0 or epoch == 0:
        print(f"{epoch+1:<8} {train_loss:<12.4f} {val_loss:<12.4f} {val_acc:<10.4f}")
    
    # Early stopping
    if val_acc > history['best_val_acc']:
        history['best_val_acc'] = val_acc
        history['best_epoch'] = epoch + 1
        patience_counter = 0
        # Save best model
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"⏹️  Early stopping at epoch {epoch+1}")
            model.load_state_dict(best_model_state)
            break

print(f"\n✅ Training complete!")
print(f"  Best epoch: {history['best_epoch']}")
print(f"  Best val accuracy: {history['best_val_acc']:.4f}")

## Step 5: Evaluate Model

In [ ]:
# Cell 7: Evaluate on Test Set

print("📈 Evaluating on test set...")

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(DEVICE)
        logits = model(batch_x)
        preds = torch.argmax(logits, dim=1) + 1  # Convert to 1-10
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend((batch_y.numpy() + 1))  # Convert to 1-10

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds, labels=range(1, 11))

print(f"\n✅ Test Accuracy: {accuracy:.4f}")
print(f"\n📊 Per-class Performance:")
print(classification_report(all_labels, all_preds, labels=range(1, 11), zero_division=0))

# Save predictions
results_df = pd.DataFrame({
    'true_label': all_labels,
    'prediction': all_preds
})

print(f"\n📋 Predictions:")
print(results_df)

## Step 6: Visualizations

In [ ]:
# Cell 8: Plot Training Curves

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history['val_acc'], label='Val Accuracy', marker='o', color='green')
axes[1].axhline(y=history['best_val_acc'], color='r', linestyle='--', 
                label=f"Best: {history['best_val_acc']:.4f}")
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/output/training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Training curves saved to /kaggle/output/training_curves.png")

In [ ]:
# Cell 9: Plot Confusion Matrix

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(1, 11), yticklabels=range(1, 11))
ax.set_xlabel('Predicted NoC')
ax.set_ylabel('True NoC')
ax.set_title(f'Confusion Matrix (Accuracy: {accuracy:.4f})')
plt.tight_layout()
plt.savefig('/kaggle/output/confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved to /kaggle/output/confusion_matrix.png")

## Step 7: Save Results

In [ ]:
# Cell 10: Save Model and Results

print("💾 Saving results...")

# Save model
torch.save(model.state_dict(), '/kaggle/output/model.pt')
print("✅ Model saved to /kaggle/output/model.pt")

# Save metrics
metrics = {
    'test_accuracy': float(accuracy),
    'best_val_accuracy': float(history['best_val_acc']),
    'best_epoch': history['best_epoch'],
    'config': {
        'hidden_dim': 128,
        'num_classes': 10,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS
    }
}

with open('/kaggle/output/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("✅ Metrics saved to /kaggle/output/metrics.json")

# Save predictions
results_df.to_csv('/kaggle/output/predictions.csv', index=False)
print("✅ Predictions saved to /kaggle/output/predictions.csv")

# Save feature names
with open('/kaggle/output/feature_names.json', 'w') as f:
    json.dump(feature_names, f, indent=2)
print("✅ Feature names saved to /kaggle/output/feature_names.json")

print(f"\n{'='*60}")
print(f"🎉 Training Complete!")
print(f"{'='*60}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Best Epoch: {history['best_epoch']}")
print(f"\nOutput files in /kaggle/output/:")
print(f"  - model.pt (trained model)")
print(f"  - metrics.json (performance metrics)")
print(f"  - predictions.csv (test predictions)")
print(f"  - training_curves.png (loss & accuracy plots)")
print(f"  - confusion_matrix.png (confusion matrix)")
print(f"  - feature_names.json (feature names)")
print(f"{'='*60}")